# 🎥 Video Deepfake Detection — Fine-Tuning Pipeline

## AuthentiQ AI Content Intelligence Platform

Based on the [Kaggle Deepfake Detection Challenge](https://www.kaggle.com/c/deepfake-detection-challenge) approach:

**Architecture:**
- **Face Extraction:** MTCNN (Multi-Task Cascaded Convolutional Networks)
- **Backbone:** EfficientNet-B0 (pre-trained on ImageNet) → fine-tuned for binary classification
- **Pipeline:** Video → Frame Sampling → Face Crop → EfficientNet → Real/Fake

**This notebook covers:**
1. Environment setup & dependency installation
2. MTCNN face extraction from video frames
3. Custom PyTorch Dataset for deepfake videos
4. EfficientNet-B0 fine-tuning (freeze backbone → train head → unfreeze & fine-tune)
5. Training loop with mixed precision & scheduling
6. Evaluation (ROC-AUC, confusion matrix, per-video aggregation)
7. Export model for inference via `video_detector.py`

**Reference:** [Kaggle Deepfake Detection by Muhammad Husban](https://www.kaggle.com/code/muhammadhusban/deepfake-detection/notebook)

In [ ]:
# ── Cell 1: Install Dependencies ─────────────────────────────────────────────
import subprocess, sys

packages = [
    'torch', 'torchvision', 'torchaudio',
    'facenet-pytorch',           # MTCNN face detection
    'efficientnet-pytorch',      # EfficientNet backbone
    'opencv-python-headless',    # Video frame extraction
    'albumentations',            # Advanced augmentations
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'tqdm',
    'Pillow',
]

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ All dependencies installed')

In [ ]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os
import cv2
import json
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T

from PIL import Image
from facenet_pytorch import MTCNN
from efficientnet_pytorch import EfficientNet

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, classification_report,
    confusion_matrix, roc_curve, precision_recall_curve, average_precision_score,
)

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'🖥️  Device: {device}')
print(f'🔦 PyTorch: {torch.__version__}')
print(f'📷 OpenCV:  {cv2.__version__}')

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────────────────

# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PATHS to match your dataset location               ║
# ╚══════════════════════════════════════════════════════════════════╝

# For Kaggle DFDC dataset:
#   VIDEO_DIR should point to the folder containing .mp4 files
#   METADATA_FILE should point to metadata.json with {filename: {label: REAL/FAKE}}

PROJECT_ROOT = Path('../../..')                       # ai/ml/notebooks → project root
VIDEO_DIR    = PROJECT_ROOT / 'dataset' / 'videos'    # Folder with .mp4 files
METADATA_FILE = PROJECT_ROOT / 'dataset' / 'metadata.json'  # DFDC metadata
MODELS_DIR   = Path('../models')                      # Save trained models here
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Hyperparameters
CFG = {
    'model_name':     'efficientnet-b0',   # Backbone (b0 for speed, b4/b7 for accuracy)
    'image_size':     224,                  # EfficientNet-B0 input size
    'n_frames':       16,                   # Frames to sample per video
    'batch_size':     16,                   # Batch size (faces, not videos)
    'epochs_head':    3,                    # Phase 1: train only classification head
    'epochs_finetune': 5,                   # Phase 2: unfreeze & fine-tune full network
    'lr_head':        1e-3,                 # LR for head-only training
    'lr_finetune':    3e-5,                 # LR for full fine-tuning
    'weight_decay':   1e-4,
    'margin':         40,                   # MTCNN face crop margin (pixels)
    'min_face_size':  60,                   # Minimum face size to detect
    'val_split':      0.15,
    'num_workers':    0,                    # 0 for macOS, 4 for Linux/Kaggle
}

print('📋 Configuration:')
for k, v in CFG.items():
    print(f'   {k}: {v}')

---
## 1. MTCNN Face Extraction Pipeline

We extract faces from video frames using MTCNN (Multi-Task Cascaded Convolutional Networks).
This is the same approach used in the Kaggle DFDC competition top solutions.

**Pipeline:** Video → Sample N frames → Detect faces → Crop & align → Save as images

In [ ]:
# ── Cell 4: MTCNN Face Extractor ─────────────────────────────────────────────

class VideoFaceExtractor:
    """
    Extract face crops from video files using MTCNN.
    
    For each video:
    1. Sample n_frames evenly across the video duration
    2. Run MTCNN face detection on each frame
    3. Crop, align, and resize the detected face
    4. Return list of PIL Images (face crops)
    """
    
    def __init__(self, n_frames=16, image_size=224, margin=40, min_face_size=60):
        self.n_frames = n_frames
        self.image_size = image_size
        self.detector = MTCNN(
            image_size=image_size,
            margin=margin,
            min_face_size=min_face_size,
            thresholds=[0.6, 0.7, 0.7],
            factor=0.709,
            post_process=False,     # Don't normalize — we do it ourselves
            device=device,
        )
    
    def extract_frames(self, video_path):
        """Sample n_frames evenly from the video."""
        cap = cv2.VideoCapture(str(video_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total <= 0:
            cap.release()
            return []
        
        indices = np.linspace(0, total - 1, min(self.n_frames, total), dtype=int)
        frames = []
        
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame_rgb))
        
        cap.release()
        return frames
    
    def detect_faces(self, frames):
        """Run MTCNN on a list of PIL Images, return face crop tensors."""
        face_crops = []
        
        for frame in frames:
            try:
                # MTCNN returns a tensor of shape (3, image_size, image_size) or None
                face = self.detector(frame)
                if face is not None:
                    # face is a tensor (3, 224, 224) with values in [0, 255]
                    face_crops.append(face)
            except Exception:
                continue
        
        return face_crops
    
    def __call__(self, video_path):
        """Extract face crops from a video file."""
        frames = self.extract_frames(video_path)
        if not frames:
            return []
        return self.detect_faces(frames)


# Initialize extractor
extractor = VideoFaceExtractor(
    n_frames=CFG['n_frames'],
    image_size=CFG['image_size'],
    margin=CFG['margin'],
    min_face_size=CFG['min_face_size'],
)

print('✅ MTCNN Face Extractor initialized')
print(f'   Sampling {CFG["n_frames"]} frames per video')
print(f'   Face crop size: {CFG["image_size"]}×{CFG["image_size"]}')

---
## 2. Dataset Preparation

We support two dataset formats:
- **DFDC format:** metadata.json with `{"filename.mp4": {"label": "REAL"}}` 
- **Folder format:** `real/` and `fake/` subdirectories containing video files
- **Demo mode:** Generate synthetic training data if no dataset is present

In [ ]:
# ── Cell 5: Load Dataset Metadata ────────────────────────────────────────────

def load_dfdc_metadata(metadata_path, video_dir):
    """Load DFDC-style metadata.json → DataFrame with columns [video_path, label]."""
    with open(metadata_path) as f:
        meta = json.load(f)
    
    rows = []
    for filename, info in meta.items():
        vpath = Path(video_dir) / filename
        if vpath.exists():
            label = 1 if info.get('label', '').upper() == 'FAKE' else 0
            rows.append({'video_path': str(vpath), 'label': label, 'filename': filename})
    
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} videos from DFDC metadata')
    print(f'  Real: {(df.label == 0).sum()} | Fake: {(df.label == 1).sum()}')
    return df


def load_folder_dataset(video_dir):
    """Load from real/ and fake/ subdirectories."""
    rows = []
    video_dir = Path(video_dir)
    
    for label_name, label_int in [('real', 0), ('fake', 1)]:
        folder = video_dir / label_name
        if folder.exists():
            for f in folder.glob('*.mp4'):
                rows.append({'video_path': str(f), 'label': label_int, 'filename': f.name})
            for f in folder.glob('*.avi'):
                rows.append({'video_path': str(f), 'label': label_int, 'filename': f.name})
    
    df = pd.DataFrame(rows)
    if len(df) > 0:
        print(f'Loaded {len(df)} videos from folder structure')
        print(f'  Real: {(df.label == 0).sum()} | Fake: {(df.label == 1).sum()}')
    return df


def create_demo_dataset(n_samples=200):
    """
    Create a synthetic demo dataset using random face-like tensors.
    This allows the full pipeline to run even without real video data.
    For production, replace with actual DFDC or FaceForensics++ data.
    """
    print(f'⚠️  No video dataset found. Creating synthetic demo dataset ({n_samples} samples)...')
    print('   For real training, download DFDC dataset from Kaggle and set VIDEO_DIR.')
    
    demo_dir = MODELS_DIR / 'demo_faces'
    demo_dir.mkdir(exist_ok=True)
    
    rows = []
    for i in range(n_samples):
        label = i % 2  # Alternate real/fake
        
        # Generate synthetic face-like image with distinguishing features
        img = np.random.randint(100, 200, (224, 224, 3), dtype=np.uint8)
        if label == 1:  # Fake: add artifacts (blending edges, noise)
            img[80:180, 80:180] = np.random.randint(50, 150, (100, 100, 3), dtype=np.uint8)
            # Add Gaussian noise (simulating compression artifacts)
            noise = np.random.normal(0, 25, img.shape).astype(np.int16)
            img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        else:  # Real: smoother, more natural gradients
            for c in range(3):
                img[:, :, c] = cv2.GaussianBlur(img[:, :, c], (15, 15), 5)
        
        fpath = demo_dir / f'face_{i:04d}.jpg'
        cv2.imwrite(str(fpath), img)
        rows.append({'face_path': str(fpath), 'label': label, 'filename': f'face_{i:04d}.jpg'})
    
    df = pd.DataFrame(rows)
    print(f'✅ Created {len(df)} synthetic face images')
    print(f'   Real: {(df.label == 0).sum()} | Fake: {(df.label == 1).sum()}')
    return df


# Try loading in order of preference
USE_DEMO = False
if METADATA_FILE.exists():
    df_all = load_dfdc_metadata(METADATA_FILE, VIDEO_DIR)
elif VIDEO_DIR.exists():
    df_all = load_folder_dataset(VIDEO_DIR)
    if len(df_all) == 0:
        df_all = create_demo_dataset()
        USE_DEMO = True
else:
    df_all = create_demo_dataset()
    USE_DEMO = True

print(f'\n📊 Dataset size: {len(df_all)}')

In [ ]:
# ── Cell 6: Train/Val Split ──────────────────────────────────────────────────

df_train, df_val = train_test_split(
    df_all, test_size=CFG['val_split'], random_state=SEED, stratify=df_all['label']
)

print(f'Train: {len(df_train)} samples (Real: {(df_train.label==0).sum()}, Fake: {(df_train.label==1).sum()})')
print(f'Val:   {len(df_val)} samples (Real: {(df_val.label==0).sum()}, Fake: {(df_val.label==1).sum()})')

In [ ]:
# ── Cell 7: PyTorch Dataset ──────────────────────────────────────────────────

# Image normalization (ImageNet statistics for EfficientNet)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.RandomGrayscale(p=0.05),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = T.Compose([
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class DeepfakeVideoDataset(Dataset):
    """
    Dataset that loads face crops from videos (or pre-extracted face images).
    
    For video mode: extracts faces on-the-fly using MTCNN
    For demo/image mode: loads pre-saved face crops from disk
    """
    
    def __init__(self, df, extractor=None, transform=None, use_demo=False):
        self.df = df.reset_index(drop=True)
        self.extractor = extractor
        self.transform = transform
        self.use_demo = use_demo
        
        if use_demo:
            # Pre-extract mode: load from face_path column
            self.samples = [
                (row['face_path'], row['label'])
                for _, row in self.df.iterrows()
            ]
        else:
            # Video mode: extract faces on-the-fly
            self.samples = [
                (row['video_path'], row['label'])
                for _, row in self.df.iterrows()
            ]
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        
        if self.use_demo:
            # Load pre-saved face image
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            face_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        else:
            # Extract faces from video
            faces = self.extractor(path)
            if len(faces) > 0:
                # Pick random face frame during training
                face_tensor = random.choice(faces).float() / 255.0
            else:
                # Fallback: black image if no face detected
                face_tensor = torch.zeros(3, 224, 224)
        
        if self.transform:
            face_tensor = self.transform(face_tensor)
        
        return face_tensor, torch.tensor(label, dtype=torch.float32)


# Create datasets
train_ds = DeepfakeVideoDataset(df_train, extractor=extractor, transform=train_transforms, use_demo=USE_DEMO)
val_ds   = DeepfakeVideoDataset(df_val,   extractor=extractor, transform=val_transforms,   use_demo=USE_DEMO)

# Handle class imbalance with weighted sampling
train_labels = [s[1] for s in train_ds.samples]
class_counts = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_weights = [class_weights[l] for l in train_labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler, num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,  num_workers=CFG['num_workers'], pin_memory=True)

print(f'✅ DataLoaders created')
print(f'   Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

---
## 3. Model Architecture — EfficientNet Fine-Tuning

We use a **two-phase fine-tuning** strategy:

1. **Phase 1 (Head-only):** Freeze the EfficientNet backbone, train only the new classification head
2. **Phase 2 (Full fine-tune):** Unfreeze all layers, train end-to-end with a lower learning rate

This prevents catastrophic forgetting of the pre-trained ImageNet features.

In [ ]:
# ── Cell 8: Define DeepfakeDetector Model ────────────────────────────────────

class DeepfakeDetector(nn.Module):
    """
    EfficientNet-based deepfake detector with a custom classification head.
    
    Architecture:
        EfficientNet-B0 backbone (frozen initially)
        → AdaptiveAvgPool → Dropout → FC(1280, 512) → ReLU → Dropout → FC(512, 1)
    """
    
    def __init__(self, model_name='efficientnet-b0', pretrained=True, dropout=0.3):
        super().__init__()
        
        # Load pretrained EfficientNet backbone
        if pretrained:
            self.backbone = EfficientNet.from_pretrained(model_name)
        else:
            self.backbone = EfficientNet.from_name(model_name)
        
        # Get the feature dimension from the backbone
        in_features = self.backbone._fc.in_features  # 1280 for B0
        
        # Replace the default classifier with our custom head
        self.backbone._fc = nn.Identity()
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(p=dropout / 2),
            nn.Linear(512, 1),
        )
        
        self._in_features = in_features
    
    def freeze_backbone(self):
        """Freeze all backbone parameters — train only the classifier head."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        print('🔒 Backbone frozen — training head only')
    
    def unfreeze_backbone(self):
        """Unfreeze all backbone parameters for full fine-tuning."""
        for param in self.backbone.parameters():
            param.requires_grad = True
        print('🔓 Backbone unfrozen — full fine-tuning enabled')
    
    def forward(self, x):
        features = self.backbone(x)     # (B, 1280)
        logits = self.classifier(features)  # (B, 1)
        return logits.squeeze(1)         # (B,)


# Instantiate model
model = DeepfakeDetector(
    model_name=CFG['model_name'],
    pretrained=True,
    dropout=0.3,
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n✅ DeepfakeDetector loaded')
print(f'   Backbone:         {CFG["model_name"]}')
print(f'   Total params:     {total_params:,}')
print(f'   Trainable params: {trainable:,}')

In [ ]:
# ── Cell 9: Training Utilities ───────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    """Train for one epoch with mixed precision."""
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    pbar = tqdm(loader, desc='  Training', leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        if device.type == 'cuda':
            with autocast():
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        
        # Track metrics
        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix(loss=f'{loss.item():.4f}')
    
    epoch_loss = running_loss / len(loader.dataset)
    try:
        epoch_auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        epoch_auc = 0.5
    epoch_acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])
    
    return epoch_loss, epoch_auc, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Evaluate on validation set."""
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    
    for images, labels in tqdm(loader, desc='  Validating', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        logits = model(images)
        loss = criterion(logits, labels)
        
        running_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_preds.extend(probs)
        all_labels.extend(labels.cpu().numpy())
    
    val_loss = running_loss / len(loader.dataset)
    try:
        val_auc = roc_auc_score(all_labels, all_preds)
    except ValueError:
        val_auc = 0.5
    val_acc = accuracy_score(all_labels, [1 if p > 0.5 else 0 for p in all_preds])
    
    return val_loss, val_auc, val_acc, all_preds, all_labels

print('✅ Training utilities defined')

---
## 4. Two-Phase Fine-Tuning

### Phase 1: Head-only training (backbone frozen)
Only the classification head is trained while EfficientNet backbone weights stay fixed.

In [ ]:
# ── Cell 10: Phase 1 — Train Classification Head ─────────────────────────────

print('═' * 60)
print('Phase 1: Head-Only Training (Backbone Frozen)')
print('═' * 60)

model.freeze_backbone()

trainable_phase1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable_phase1:,} / {total_params:,}')

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CFG['lr_head'],
    weight_decay=CFG['weight_decay'],
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs_head'])
scaler = GradScaler(enabled=(device.type == 'cuda'))

history = {'train_loss': [], 'val_loss': [], 'train_auc': [], 'val_auc': [], 'train_acc': [], 'val_acc': []}
best_auc = 0.0

t0 = time.time()
for epoch in range(CFG['epochs_head']):
    print(f'\nEpoch {epoch+1}/{CFG["epochs_head"]}')
    
    train_loss, train_auc, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, val_auc, val_acc, _, _ = validate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f'  Train — Loss: {train_loss:.4f} | AUC: {train_auc:.4f} | Acc: {train_acc:.4f}')
    print(f'  Val   — Loss: {val_loss:.4f} | AUC: {val_auc:.4f} | Acc: {val_acc:.4f}')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), MODELS_DIR / 'deepfake_best_head.pth')
        print(f'  💾 Saved best head model (AUC: {best_auc:.4f})')

print(f'\n✅ Phase 1 complete in {time.time()-t0:.1f}s | Best Val AUC: {best_auc:.4f}')

### Phase 2: Full Fine-Tuning (backbone unfrozen)

Now we unfreeze all EfficientNet layers and fine-tune end-to-end with a **much lower learning rate** to preserve the pre-trained features.

In [ ]:
# ── Cell 11: Phase 2 — Full Fine-Tuning ──────────────────────────────────────

print('═' * 60)
print('Phase 2: Full Fine-Tuning (All Layers)')
print('═' * 60)

# Load best head weights
model.load_state_dict(torch.load(MODELS_DIR / 'deepfake_best_head.pth', map_location=device, weights_only=True))
model.unfreeze_backbone()

trainable_phase2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable_phase2:,} / {total_params:,}')

# Differential learning rates: backbone gets lower LR
backbone_params = [p for n, p in model.named_parameters() if 'classifier' not in n and p.requires_grad]
head_params     = [p for n, p in model.named_parameters() if 'classifier' in n and p.requires_grad]

optimizer = optim.AdamW([
    {'params': backbone_params, 'lr': CFG['lr_finetune']},
    {'params': head_params,     'lr': CFG['lr_finetune'] * 10},  # Head gets 10x higher LR
], weight_decay=CFG['weight_decay'])

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2)
scaler = GradScaler(enabled=(device.type == 'cuda'))

t0 = time.time()
for epoch in range(CFG['epochs_finetune']):
    print(f'\nEpoch {epoch+1}/{CFG["epochs_finetune"]}')
    
    train_loss, train_auc, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
    val_loss, val_auc, val_acc, _, _ = validate(model, val_loader, criterion, device)
    scheduler.step(epoch)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_auc'].append(train_auc)
    history['val_auc'].append(val_auc)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    print(f'  Train — Loss: {train_loss:.4f} | AUC: {train_auc:.4f} | Acc: {train_acc:.4f}')
    print(f'  Val   — Loss: {val_loss:.4f} | AUC: {val_auc:.4f} | Acc: {val_acc:.4f}')
    
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), MODELS_DIR / 'deepfake_best_finetuned.pth')
        print(f'  💾 Saved best fine-tuned model (AUC: {best_auc:.4f})')

print(f'\n✅ Phase 2 complete in {time.time()-t0:.1f}s | Best Val AUC: {best_auc:.4f}')

---
## 5. Evaluation & Visualization

In [ ]:
# ── Cell 12: Plot Training History ───────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', linewidth=2)
axes[0].axvline(x=CFG['epochs_head']-0.5, color='gray', linestyle='--', alpha=0.5, label='Phase 1→2')
axes[0].set_title('Loss', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

# AUC
axes[1].plot(history['train_auc'], label='Train', linewidth=2)
axes[1].plot(history['val_auc'], label='Val', linewidth=2)
axes[1].axvline(x=CFG['epochs_head']-0.5, color='gray', linestyle='--', alpha=0.5, label='Phase 1→2')
axes[1].set_title('ROC-AUC', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Accuracy
axes[2].plot(history['train_acc'], label='Train', linewidth=2)
axes[2].plot(history['val_acc'], label='Val', linewidth=2)
axes[2].axvline(x=CFG['epochs_head']-0.5, color='gray', linestyle='--', alpha=0.5, label='Phase 1→2')
axes[2].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.suptitle('Deepfake Detection — Training History', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'deepfake_training_history.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved training history plot')

In [ ]:
# ── Cell 13: Final Evaluation ────────────────────────────────────────────────

# Load best model
best_path = MODELS_DIR / 'deepfake_best_finetuned.pth'
if not best_path.exists():
    best_path = MODELS_DIR / 'deepfake_best_head.pth'

model.load_state_dict(torch.load(best_path, map_location=device, weights_only=True))
val_loss, val_auc, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)

print('═' * 60)
print('Final Evaluation on Validation Set')
print('═' * 60)
print(f'Loss:     {val_loss:.4f}')
print(f'ROC-AUC:  {val_auc:.4f}')
print(f'Accuracy: {val_acc:.4f}')
print()

# Classification report
val_pred_labels = [1 if p > 0.5 else 0 for p in val_preds]
print(classification_report(val_labels, val_pred_labels, target_names=['Real', 'Fake']))

In [ ]:
# ── Cell 14: ROC Curve & Confusion Matrix ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
fpr, tpr, thresholds = roc_curve(val_labels, val_preds)
axes[0].plot(fpr, tpr, linewidth=2, color='#6366f1')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#6366f1')
axes[0].set_title(f'ROC Curve (AUC = {val_auc:.4f})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].grid(alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(val_labels, val_preds)
ap = average_precision_score(val_labels, val_preds)
axes[1].plot(recall, precision, linewidth=2, color='#f59e0b')
axes[1].fill_between(recall, precision, alpha=0.1, color='#f59e0b')
axes[1].set_title(f'Precision-Recall (AP = {ap:.4f})', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].grid(alpha=0.3)

# Confusion Matrix
cm = confusion_matrix(val_labels, val_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
axes[2].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.suptitle('Deepfake Detection — Evaluation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'deepfake_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved evaluation plots')

---
## 6. Export Model for Production

Save the fine-tuned model along with metadata for the `video_detector.py` inference module.

In [ ]:
# ── Cell 15: Export Final Model & Metadata ───────────────────────────────────

# Save the complete model state
final_path = MODELS_DIR / 'deepfake_efficientnet_b0.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'model_name': CFG['model_name'],
    'image_size': CFG['image_size'],
    'n_frames': CFG['n_frames'],
    'margin': CFG['margin'],
    'min_face_size': CFG['min_face_size'],
}, final_path)

# Save metadata
deepfake_meta = {
    'model_version': '1.0.0',
    'model_name': CFG['model_name'],
    'architecture': 'EfficientNet-B0 + Custom Head',
    'task': 'video_deepfake_detection',
    'image_size': CFG['image_size'],
    'n_frames': CFG['n_frames'],
    'margin': CFG['margin'],
    'min_face_size': CFG['min_face_size'],
    'val_auc': round(val_auc, 4),
    'val_accuracy': round(val_acc, 4),
    'threshold': 0.5,
    'training_phases': [
        f'Phase 1: Head-only ({CFG["epochs_head"]} epochs, LR={CFG["lr_head"]})',
        f'Phase 2: Full fine-tune ({CFG["epochs_finetune"]} epochs, LR={CFG["lr_finetune"]})',
    ],
    'face_detector': 'MTCNN (facenet-pytorch)',
    'backbone_pretrained_on': 'ImageNet',
    'reference': 'https://www.kaggle.com/code/muhammadhusban/deepfake-detection/notebook',
}

with open(MODELS_DIR / 'deepfake_meta.json', 'w') as f:
    json.dump(deepfake_meta, f, indent=2)

print('✅ Model exported successfully')
print(f'   Model weights: {final_path}')
print(f'   Model size:    {final_path.stat().st_size / 1024 / 1024:.1f} MB')
print(f'   Metadata:      {MODELS_DIR / "deepfake_meta.json"}')

In [ ]:
# ── Cell 16: Quick Inference Demo ────────────────────────────────────────────

def predict_single_image(model, image_path, device, transform=None):
    """Run inference on a single face image."""
    model.eval()
    
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    
    if transform:
        tensor = transform(tensor)
    
    tensor = tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        logit = model(tensor)
        prob = torch.sigmoid(logit).item()
    
    verdict = 'FAKE' if prob >= 0.5 else 'REAL'
    return prob, verdict


# Test on demo images if available
if USE_DEMO:
    demo_dir = MODELS_DIR / 'demo_faces'
    test_files = sorted(demo_dir.glob('*.jpg'))[:6]
    
    print('🔍 Quick Inference Demo:')
    print('-' * 50)
    for f in test_files:
        prob, verdict = predict_single_image(model, f, device, transform=val_transforms)
        label = 'Fake' if int(f.stem.split('_')[1]) % 2 == 1 else 'Real'
        match = '✅' if (verdict == 'FAKE') == (label == 'Fake') else '❌'
        print(f'  {f.name}: P(fake)={prob:.3f} → {verdict} (actual: {label}) {match}')

print('\n✅ Inference demo complete')

---
## 7. Summary

### What was built:

| Component | Details |
|-----------|--------|
| **Face Detection** | MTCNN — extracts & aligns faces from video frames |
| **Backbone** | EfficientNet-B0 (pre-trained on ImageNet) |
| **Fine-tuning** | 2-phase: head-only → full end-to-end |
| **Output** | Binary probability (0 = Real, 1 = Fake) |

### Files produced:

```
ai/ml/models/
├── deepfake_efficientnet_b0.pth      ← Full model checkpoint
├── deepfake_best_finetuned.pth       ← Best fine-tuned weights
├── deepfake_meta.json                ← Model metadata
├── deepfake_training_history.png     ← Training curves
└── deepfake_evaluation.png           ← ROC, PR, Confusion Matrix
```

### Next steps:
1. **Train on real data:** Download DFDC dataset from Kaggle and update `VIDEO_DIR`
2. **Use the inference module:** `from ai.ml.video_detector import get_video_detector`
3. **Wire into FastAPI:** The `video_detector.py` module exposes the same API pattern as `detector.py`

### Reference:
- [Kaggle DFDC Competition](https://www.kaggle.com/c/deepfake-detection-challenge)
- [Muhammad Husban's Notebook](https://www.kaggle.com/code/muhammadhusban/deepfake-detection/notebook)
- [EfficientNet Paper](https://arxiv.org/abs/1905.11946)